## 0. GPU 확인

# 04 — 모듈 ① 전체 파이프라인 실행 (D-3 누수차단·재현성 반영)

`docs/reproducibility_pipeline.md`의 정석 순서를 Colab에서 한 번에 실행한다.

**핵심 (이전 파이프라인 대비 바뀐 점):**
- 합성 병합 시 **MinHash 중복 제거**로 시드↔합성 누수 차단
- Stacking은 **OOF logits**만 사용(in-fold 누수 제거) — `generate_oof_logits.py`
- 모든 run **MLflow 자동 로깅**(`mlruns/`)
- 지표는 긴급(3) 포함 **4-class 고정 채점**

**런타임 요건**: GPU (A100 권장). 2·3단계가 KcELECTRA fine-tune.

> ⚠️ **3단계(OOF) 비용 주의**: 5-fold면 base fine-tune을 **5회** 더 돈다(각 fold가 train의 4/5).
> A100에서도 OOF 생성만 수 시간 걸릴 수 있다. 비용이 부담되면 아래 `OOF_FOLDS=3`으로 낮춘다.

In [1]:
!nvidia-smi | head -15
import torch
print(f"\nPyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"Device: {p.name}, VRAM: {p.total_memory/1e9:.1f} GB")
else:
    print("⚠ GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경 > GPU 선택.")

Wed Jun 10 05:42:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   60C    P8             18W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Repo Clone (private)

GitHub Personal Access Token 필요. `REPO_OWNER`를 본인 username으로 수정.
**재실행 안전**: 이미 clone돼 있으면 pull.

In [2]:
import os, getpass, subprocess
from pathlib import Path

REPO_OWNER = "threeGuineas"   # ← 본인 GitHub username으로 수정
REPO_NAME = "thisabled-ai"
BRANCH = "fix/d3-leakage-reproducibility"   # D-3 작업 브랜치
REPO_DIR = Path(f"/content/{REPO_NAME}")

if REPO_DIR.exists():
    print(f"Repo exists at {REPO_DIR} — fetch & checkout {BRANCH}")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub PAT: ")
    url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO_DIR)], check=True)
    print(f"Cloned ({BRANCH}) to {REPO_DIR}")

%cd {REPO_DIR}
!git log --oneline -3

Cloned (fix/d3-leakage-reproducibility) to /content/thisabled-ai
/content/thisabled-ai
974179d (HEAD -> fix/d3-leakage-reproducibility, origin/fix/d3-leakage-reproducibility) fix(D-2): evaluate_real_holdout 체크포인트 부재 시 명확한 에러 + 상대경로 보정
a6b3a37 feat(D-2): 비순환 실데이터 hold-out (AI-Hub 실긴급 + BEEP) 평가 경로
3b2460b data(D-3): track synthetic emergency jsonl via gitignore exception


## 2. 의존성 설치

`requirements-colab.txt`에 `mlflow`·`datasketch`가 포함돼 있어야 D-3 단계가 동작한다.

In [3]:
!pip install -q -r requirements-colab.txt
!pip install -q "accelerate>=0.26"
import importlib
for m in ("mlflow", "datasketch"):
    print(m, "OK" if importlib.util.find_spec(m) else "❌ 누락 — requirements-colab.txt 확인")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.7/34.7 MB 74.4 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 155.4 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.4/364.4 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.3/232.3 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 101.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 105.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 3. 파이프라인 파라미터

한 곳에서 노브를 설정하고 이후 셀에서 환경변수로 전달한다.

In [4]:
import os
os.environ["CONFIG"]       = "configs/module1_kcelectra_ce.yaml"
os.environ["SYNTH_REPEAT"] = "8"     # 합성 oversample 배수 (0이면 합성 미포함)
os.environ["OOF_FOLDS"]    = "5"     # OOF Stratified fold 수 (비용 부담시 3)
os.environ["MODEL_DIR"]    = "models/checkpoints/module1_ce"  # config paths.checkpoint_dir와 일치
print({k: os.environ[k] for k in ("CONFIG","SYNTH_REPEAT","OOF_FOLDS","MODEL_DIR")})

{'CONFIG': 'configs/module1_kcelectra_ce.yaml', 'SYNTH_REPEAT': '8', 'OOF_FOLDS': '5', 'MODEL_DIR': 'models/checkpoints/module1_ce'}


### 3-1. 데이터 빌드 (시드 다운로드 → processed → 합성 병합 + MinHash 중복 제거)

`build_final_dataset.py`가 합성 train을 시드와 비교해 근사 중복(Jaccard≥0.8)을 제거한 뒤 oversample 한다.

In [5]:
!python scripts/download_seed_datasets.py
!python scripts/build_processed_dataset.py
!python scripts/build_final_dataset.py --synth-repeat $SYNTH_REPEAT

target: /content/thisabled-ai/data/raw
[unsmile/train] downloading
  -> https://raw.githubusercontent.com/smilegate-ai/korean_unsmile_dataset/main/unsmile_train_v1.0.tsv
[unsmile/train] ok (1,848,905 bytes, 15,005 rows)
[unsmile/valid] downloading
  -> https://raw.githubusercontent.com/smilegate-ai/korean_unsmile_dataset/main/unsmile_valid_v1.0.tsv
[unsmile/valid] ok (455,315 bytes, 3,737 rows)
[kold] downloading (LFS)
  -> https://media.githubusercontent.com/media/boychaboy/KOLD/main/data/kold_v1.json
[kold] ok (32,791,434 bytes, sha256 verified)
done.
=== Splits saved ===
[train] n=47,336  labels={0: 19836, 1: 9460, 2: 18040}  sources={'kold': 32291, 'unsmile': 15045}  -> data/processed/train.parquet
[val] n=5,917  labels={0: 2479, 1: 1183, 2: 2255}  sources={'kold': 4111, 'unsmile': 1806}  -> data/processed/val.parquet
[test] n=5,918  labels={0: 2480, 1: 1183, 2: 2255}  sources={'kold': 4027, 'unsmile': 1891}  -> data/processed/test.parquet
=== 통합 최종 데이터셋 빌드 (synth_repeat=8) ===
1. 

### 3-2. base KcELECTRA fine-tune (MLflow 자동 로깅)

In [6]:
!python scripts/train_module1.py --config $CONFIG

2026-06-10 05:45:59.803986: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-10 05:45:59.877578: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 100% 288/288 [00:00<00:00, 2.42MB/s]
config.json: 100% 504/504 [00:00<00:00, 5.23MB/s]
vocab.txt: 450kB [00:00, 69.0MB/s]
special_tokens_map.json: 100% 124/124 [00:00<00:00, 1.44MB/s]
pytorch_model.bin: 100% 511M/511M [02:03<00:00, 4.15MB/s]
Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcEL

### 3-3. OOF logits 생성 (Stratified K-fold, seed=42)

⚠️ **여기가 비용 큰 단계** — fold 수만큼 base를 새로 fine-tune 한다.
산출: `data/processed/oof_train_logits.npy` (train.parquet 행과 정렬).

In [7]:
!python scripts/generate_oof_logits.py --config $CONFIG --folds $OOF_FOLDS

2026-06-10 05:58:23.830557: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-10 05:58:23.903734: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=== OOF logits 생성: 53736행, 5-fold (seed=42) ===

--- Fold 1/5: train 42988 / oof 10748 ---
    oof fold 클래스 비율: {0: 0.383, 1: 0.184, 2: 0.344, 3: 0.089}
Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base-v2022 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out

### 3-4. Stacking 메타러너 (train=OOF, val/test=full-train base)

In [8]:
!python scripts/train_stacking.py --config $CONFIG --model-dir $MODEL_DIR

2026-06-10 06:39:23.542984: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-10 06:39:23.613696: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
사용 장비: cuda
  - train: OOF logits 사용 (oof_train_logits.npy, shape=(53736, 4))
  - 모델 로딩 중: module1_ce
  - Logits 추출 중 (val.parquet)...
Inference: 100% 93/93 [00:10<00:00,  9.00it/s]
  - 모델 로딩 중: module1_ce
  - Logits 추출 중 (test.parquet)...
Inference: 100% 93/93 [00:10<00:00,  9.21it/s]

  - LightGBM Stacking 모델 훈련 시작 (train=OOF)...
/usr/local/lib/python3.12/dist-packages/

### 3-5. 합성 hold-out 평가

⚠️ 이 Recall은 **합성→합성 순환** 평가다. 실데이터 긴급 일반화를 보장하지 않는다(D-2에서 실데이터 hold-out 구성 예정).

In [9]:
!python scripts/evaluate_synthetic.py --model-dir $MODEL_DIR

2026-06-10 06:40:01.846790: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-10 06:40:01.915362: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=== [모듈 ①] 합성 Hold-out (긴급) 평가 시작 ===
1. 모델 로딩 (cuda)...
2. 합성 데이터 로딩 (synthetic_holdout.parquet)...
3. 예측 진행...

[합성 Hold-out 평가 결과]
  - 총 샘플 수: 370
  - 긴급(3) Recall: 1.0000
  - Boundary FPR: 0.0000
  ⚠ 주의: 이 Recall은 합성 hold-out 기준입니다. 합성으로 학습 후 같은 분포의 합성으로 평가하는 순환 구조이므로 실데이터 긴급 일반화를 보장하지 않습니다.

✔ 합성 평가 리포트 저장 완료: reports/validation_reports/module1/emergency_synthetic_ev

> **대안 (한 번에 실행)**: 위 3-1~3-5를 셀별로 돌리는 대신, 아래 한 줄로 정석 순서를 일괄 실행할 수 있다.
> ```
> !bash scripts/run_full_pipeline.sh
> ```

## 4. MLflow 결과 확인

기록된 run의 핵심 지표를 표로 출력한다.

In [10]:
import mlflow, pandas as pd
mlflow.set_tracking_uri("mlruns")
exp = mlflow.get_experiment_by_name("thisabled-module1")
if exp is None:
    print("아직 기록된 실험이 없습니다 — 위 단계를 먼저 실행하세요.")
else:
    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    cols = [c for c in runs.columns
            if c.startswith("metrics.") or c in ("tags.mlflow.runName", "start_time")]
    show = runs[cols].sort_values("start_time", ascending=False)
    pd.set_option("display.max_columns", None, "display.width", 200)
    display(show.head(20))
    print("\nKPI 대비: 긴급 Recall≥0.80, Macro-F1≥0.75 (긴급 클래스 포함 4-class 채점 기준)")

,start_time,metrics.holdout_boundary_fpr,metrics.holdout_emergency_recall,metrics.n_total,metrics.test_macro_f1,metrics.test_emergency_support,metrics.test_emergency_recall,metrics.test_auc_pr,metrics.final_eval_f1_class0,metrics.final_eval_f1_class1,metrics.eval_f1_class1,metrics.learning_rate,metrics.eval_macro_f1,metrics.grad_norm,metrics.final_eval_f1_class2,metrics.eval_steps_per_second,metrics.loss,metrics.final_eval_steps_per_second,metrics.eval_auc_pr,metrics.eval_f1_class2,metrics.eval_f1_class0,metrics.train_runtime,metrics.eval_loss,metrics.final_eval_f1_class3,metrics.final_epoch,metrics.final_eval_loss,metrics.eval_emergency_recall,metrics.final_eval_emergency_recall,metrics.final_eval_runtime,metrics.final_eval_auc_pr,metrics.eval_samples_per_second,metrics.total_flos,metrics.final_eval_macro_f1,metrics.train_loss,metrics.epoch,metrics.final_eval_samples_per_second,metrics.eval_f1_class3,metrics.train_samples_per_second,metrics.train_steps_per_second,metrics.eval_runtime,tags.mlflow.runName
0,2026-06-10 06:40:08.301000+00:00,0.0,1.0,370.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,synthetic-holdout-eval/module1_ce
1,2026-06-10 06:39:54.363000+00:00,NaN,NaN,NaN,0.569657,0.0,0.0,0.850615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,stacking/module1_kcelectra_ce
2,2026-06-10 05:48:21.314000+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.82431,0.656965,0.656965,2.072310e-07,0.572403,8.584522,0.808335,23.11,0.3587,23.11,0.84776,0.808335,0.82431,587.5183,0.638463,0.0,3.0,0.638463,0.0,0.0,4.0243,0.84776,1470.313,4.508119e+15,0.572403,0.512976,3.0,1470.313,0.0,274.388,8.578,4.0243,module1_kcelectra_ce



KPI 대비: 긴급 Recall≥0.80, Macro-F1≥0.75 (긴급 클래스 포함 4-class 채점 기준)


In [11]:
!git pull origin fix/d3-leakage-reproducibility
!python scripts/evaluate_real_holdout.py --model-dir models/checkpoints/module1_ce
!echo "----- JSON -----"; cat reports/validation_reports/module1/real_holdout_eval.json


remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 8 (delta 4), reused 8 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 3.38 KiB | 1.69 MiB/s, done.
From https://github.com/threeGuineas/thisabled-ai
 * branch            fix/d3-leakage-reproducibility -> FETCH_HEAD
   974179d..5ec37fc  fix/d3-leakage-reproducibility -> origin/fix/d3-leakage-reproducibility
Updating 974179d..5ec37fc
Fast-forward
 .../validation_reports/module2/fairness_dp.json    |  17 +++
 scripts/evaluate_module2_fairness.py               | 119 +++++++++++++++++++++
 2 files changed, 136 insertions(+)
 create mode 100644 reports/validation_reports/module2/fairness_dp.json
 create mode 100644 scripts/evaluate_module2_fairness.py
2026-06-10 06:40:17.227529: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off error

In [12]:
%cd /content/thisabled-ai
!ls models/checkpoints/module1_ce 2>/dev/null && echo "있음" || echo "없음 → 재학습 필요"


/content/thisabled-ai
checkpoint-1680  config.json		  tokenizer_config.json  vocab.txt
checkpoint-3360  model.safetensors	  tokenizer.json
checkpoint-5040  special_tokens_map.json  training_args.bin
있음


In [13]:
!git pull origin fix/d3-leakage-reproducibility
!python scripts/download_seed_datasets.py
!python scripts/build_processed_dataset.py
!python scripts/build_final_dataset.py --synth-repeat 8
!python scripts/train_module1.py --config configs/module1_kcelectra_ce.yaml   # ~6분
!python scripts/evaluate_real_holdout.py --model-dir models/checkpoints/module1_ce
!echo "----- JSON -----"; cat reports/validation_reports/module1/real_holdout_eval.json


From https://github.com/threeGuineas/thisabled-ai
 * branch            fix/d3-leakage-reproducibility -> FETCH_HEAD
Already up to date.
target: /content/thisabled-ai/data/raw
[unsmile/train] skip (exists, 1,848,905 bytes)
[unsmile/valid] skip (exists, 455,315 bytes)
[kold] skip (exists, sha256 verified)
done.
=== Splits saved ===
[train] n=47,336  labels={0: 19836, 1: 9460, 2: 18040}  sources={'kold': 32291, 'unsmile': 15045}  -> data/processed/train.parquet
[val] n=5,917  labels={0: 2479, 1: 1183, 2: 2255}  sources={'kold': 4111, 'unsmile': 1806}  -> data/processed/val.parquet
[test] n=5,918  labels={0: 2480, 1: 1183, 2: 2255}  sources={'kold': 4027, 'unsmile': 1891}  -> data/processed/test.parquet
=== 통합 최종 데이터셋 빌드 (synth_repeat=8) ===
1. 합성 데이터 로딩 중... (/content/thisabled-ai/data/synthetic/emergency)
  → 합성 [train]: 800건 로드 완료
  → 합성 [val]: 150건 로드 완료
  → 합성 [test]: 220건 로드 완료
  → MinHash 중복 제거(threshold=0.8): 시드 train과 근사 중복 합성 0건 제거 → 합성 800건 잔존

[train] 시드(47336) + 합성(800×8=6400)

In [14]:
%cd /content/thisabled-ai
!git pull origin fix/d3-leakage-reproducibility
!pip install -q -r requirements-colab.txt
!python scripts/download_seed_datasets.py
!python scripts/build_processed_dataset.py
!python scripts/build_final_dataset.py --synth-repeat 8 --include-aihub-train   # ← 실데이터 투입
!python scripts/train_module1.py --config configs/module1_kcelectra_ce.yaml      # ~10분
!python scripts/evaluate_real_holdout.py --model-dir models/checkpoints/module1_ce
!echo '----- JSON -----'; cat reports/validation_reports/module1/real_holdout_eval.json


/content/thisabled-ai
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 9 (delta 6), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 664.30 KiB | 8.20 MiB/s, done.
From https://github.com/threeGuineas/thisabled-ai
 * branch            fix/d3-leakage-reproducibility -> FETCH_HEAD
   5ec37fc..9fb82e0  fix/d3-leakage-reproducibility -> origin/fix/d3-leakage-reproducibility
Updating 5ec37fc..9fb82e0
Fast-forward
 .pre-commit-config.yaml        |     2 +-
 data/eval/aihub_train.jsonl    | 19979 +++++++++++++++++++++++++++++++++++++++
 scripts/build_aihub_train.py   |   111 +
 scripts/build_final_dataset.py |    82 +-
 4 files changed, 20148 insertions(+), 26 deletions(-)
 create mode 100644 data/eval/aihub_train.jsonl
 create mode 100644 scripts/build_aihub_train.py
target: /content/thisabled-ai/data/raw
[unsmile/train] skip (exists, 1,848,905 bytes)
[unsmile/valid] skip